In [1]:
import os
import cv2
import time
import random
import torch
import imageio
import ale_py
import numpy as np
import gymnasium as gym
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque, namedtuple
from torch.utils.tensorboard import SummaryWriter
from IPython.display import Image

In [2]:
gym.register_envs(ale_py)
gym.__version__

'1.3.0'

In [3]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

In [5]:
env = gym.make("Pong-v4", render_mode = 'rgb_array', frameskip =4)

A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]


In [6]:
all_obs = []
obs, info = env.reset()

for step in range(500):
    action = env.action_space.sample()
    next_state, reward, done, truncated, info = env.step(action)
    all_obs.append(env.render())
    if done or truncated:
        break
env.close()


In [ ]:
imageio.mimsave("pong_random.gif", all_obs[::2], fps=30)

In [ ]:
Image(filename="pong_random.gif")

In [13]:
def frame_preprocess(frame,width=84, height=84):
    
    frame = cv2.resize(frame[35:190],(width,height))
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return frame

In [14]:
def get_states(curr_frame, prev_state=None):
    curr_frame =  frame_preprocess(curr_frame)
    if prev_state is None:
        s = np.dstack([curr_frame, curr_frame, curr_frame, curr_frame])
    else:
        a,b,c,d = np.dsplit(prev_state, 4)
        s = np.dstack([curr_frame,a,b,c])
    return s

In [15]:
class QNetwork(nn.Module):
    
    def __init__(self, input_channels, action_size):
        super().__init__()
        self.input_channels = input_channels
        self.action_size = action_size

        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1) #7x7
        self.linear1 = nn.Linear(7*7*64, 512)
        self.linear2 = nn.Linear(512,action_size)

    def forward(self,x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.linear1(x.reshape(x.shape[0], -1)))
        x = self.linear2(x)

        return x


In [16]:
class Replay_memory:
    '''
        The memory for storing and sampling experiences
    '''
    def __init__(self, memory_size):
        self.memory_size = memory_size
        self.memory = deque(maxlen = memory_size)
        self.transition = namedtuple(typename='Transition', field_names= ['state', 'action', 'reward',\
                                                                          'next_state', 'done'])
    def add(self, state, action, reward, next_state, done):
        '''
            adding the experience to the memory
        '''
        self.memory.append(self.transition(state, action, reward, next_state, done))
        
    def sample(self, size, device):
        '''
            samples the experiences from the memory
        '''
        samples = random.sample(self.memory, size)
        states = torch.from_numpy(np.array([e.state for e in samples])).float().to(device)
        actions = torch.from_numpy(np.array([e.action for e in samples])).to(device).unsqueeze(1)
        rewards = torch.from_numpy(np.array([e.reward for e in samples])).float().to(device).unsqueeze(1)
        next_states = torch.from_numpy(np.array([e.next_state for e in samples])).float().to(device)
        dones = torch.from_numpy(np.array([e.done for e in samples])).to(device).unsqueeze(1)
        states = states.permute(0,3,1,2)
        next_states = next_states.permute(0,3,1,2)
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.memory)

In [ ]:
def epsilon_greedy_action(model: QNetwork, state: torch.Tensor, epsilon: float, env):
    '''
        epsilon greedy action
    '''
    with torch.no_grad():
        q_values = model(state)

    if random.random() < epsilon:
        action = np.random.choice(np.arange(env.action_space.n))
    else:
        action = torch.argmax(q_values, dim =1).item()

    return action

In [18]:
def update_networks(local_network: QNetwork, target_network: QNetwork, replay_memory: Replay_memory, optimizer,\
                     batch_size: int, gamma: float ,tau: float, device: torch.device):

    if len(replay_memory)>= batch_size:

        states, actions, rewards, next_states, dones = replay_memory.sample(batch_size, device)

        out = local_network(states)
        q_curr = torch.gather(out, dim=1, index= actions)

        with torch.no_grad():
            q_next = torch.max(target_network(next_states), dim = 1, keepdim=True)[0]
            target = rewards + gamma* q_next*(1-dones*1.0)

        loss = F.mse_loss(q_curr, target)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()        

        #updating the target network weights
        # target_weights = tau*local_network + (1 - tau)*target_network
        
        for local_param, target_param in zip(local_network.parameters(), target_network.parameters()):
            target_param.data.copy_(tau*local_param.data + (1-tau)*target_param.data)
        


In [ ]:
#Try to run for smaller number of episode and decrease the eps_decay, the environment
#can be trained quickly

lr = 1e-5
batch_size = 64
update_every = 8
eps_decay = .9999
eps_min = .1
gamma = 0.99
tau = 1e-3
no_episodes = 30000
replay_memory_size = int(4e5)

In [20]:
local_network = QNetwork(4,env.action_space.n).to(device)
target_network = QNetwork(4,env.action_space.n).to(device)

optimizer = torch.optim.Adam(local_network.parameters(), lr)
replay_memory = Replay_memory(replay_memory_size)

In [21]:
summary_writer = SummaryWriter('logs_pong_dqn_frame_skips')   


In [ ]:
all_scores = []
running_scores =  deque(maxlen=100)
eps = 1.0
max_score = -np.inf
for epi_no in range(no_episodes):
    epi_reward = 0
    eps = max(eps_min, eps*eps_decay)
    obs, info = env.reset()
    state = get_states(obs,None)
    update = 0
    while True:
        state_tensor = torch.tensor(state).permute(2,0,1).float().to(device).unsqueeze(0)
        action = epsilon_greedy_action(local_network, state_tensor, eps, env)
        next_obs, reward, done, truncated, info = env.step(action)
        epi_reward += np.clip(reward, -1, 1)
        if truncated or done:
             break
        next_state = get_states(next_obs, state)
        replay_memory.add(state, action, reward, next_state, done)

        update = (update+1)%update_every
        if update==0:
            update_networks(local_network, target_network, replay_memory, optimizer, batch_size, gamma, tau, device)

        state = next_state

    if epi_reward>max_score:
        max_score = epi_reward
        torch.save(local_network.state_dict(), 'checkpoint_local_pong_frame_skips.pth')

    summary_writer.add_scalar('score', epi_reward, epi_no)
    summary_writer.add_scalar('epsilon', eps, epi_no)
    if len(running_scores)>0:
        summary_writer.add_scalar('mean_score', np.mean(running_scores), epi_no)

    running_scores.append(epi_reward)
    all_scores.append(epi_reward)
    print(f'\r Episode: {epi_no} Mean score: {np.mean(running_scores):.4f} Max score: {max_score}  eps: {eps:.4f}', end="")

    if (len(running_scores)==100) and (epi_no%500==0):
        print(f'\r Episode: {epi_no} Mean score: {np.mean(running_scores):.4f} Max score: {max_score} eps: {eps:.4f}')
    

 Episode: 500 Mean score: -20.2000 Max score: -16.0 eps: 0.95111
 Episode: 1000 Mean score: -20.1100 Max score: -15.0 eps: 0.90477
 Episode: 1500 Mean score: -20.2700 Max score: -15.0 eps: 0.86066
 Episode: 2000 Mean score: -19.9200 Max score: -15.0 eps: 0.81866
 Episode: 2500 Mean score: -19.5200 Max score: -15.0 eps: 0.77877
 Episode: 3000 Mean score: -19.0700 Max score: -15.0 eps: 0.74077
 Episode: 3500 Mean score: -18.1000 Max score: -12.0 eps: 0.70466
 Episode: 4000 Mean score: -17.0300 Max score: -10.0 eps: 0.67022
 Episode: 4500 Mean score: -15.9300 Max score: -9.0 eps: 0.637660
 Episode: 5000 Mean score: -14.4600 Max score: -7.0 eps: 0.60655
 Episode: 5500 Mean score: -13.6100 Max score: -1.0 eps: 0.57699
 Episode: 6000 Mean score: -13.7000 Max score: -1.0 eps: 0.54877
 Episode: 6500 Mean score: -10.8800 Max score: 4.0 eps: 0.522005
 Episode: 7000 Mean score: -10.4600 Max score: 5.0 eps: 0.49655
 Episode: 7500 Mean score: -9.7900 Max score: 14.0 eps: 0.47233
 Episode: 8000 Mean